# Gridworld Environment Prompt Demo

This notebook walks through the prompt sequence for a short manual gridworld rollout.

It focuses on:
- the initial `GUIDE` prompt
- the `CHALLENGE` prompt after a recommendation
- the next state after resolving the challenge / move


In [ ]:
from pathlib import Path
from pprint import pprint
from types import SimpleNamespace
import importlib
import sys

NOTEBOOK_ROOT = Path.cwd().resolve()
REPO_ROOT = next((candidate for candidate in [NOTEBOOK_ROOT, *NOTEBOOK_ROOT.parents] if (candidate / 'Environments').exists() and (candidate / 'LocalizationScripts').exists()), NOTEBOOK_ROOT)
ENV_SRC = REPO_ROOT / 'Environments' / 'Gridworld' / 'src'
if str(ENV_SRC) not in sys.path:
    sys.path.insert(0, str(ENV_SRC))

import gridworld_environment as gridworld_env
importlib.reload(gridworld_env)

GridWorldEnvironment = gridworld_env.GridWorldEnvironment
GridSpec = gridworld_env.GridSpec
print('Imported from:', ENV_SRC / 'gridworld_environment.py')


In [ ]:
def make_agents():
    return [
        SimpleNamespace(name='Guide', reasoning_instruction='COD', guide_format='default', challenge_format='default'),
        SimpleNamespace(name='Explorer', reasoning_instruction='COD', guide_format='default', challenge_format='default'),
    ]


def make_env(seed=0, auto_move_explorer=False):
    return GridWorldEnvironment(
        agents=make_agents(),
        seed=seed,
        grid_spec=GridSpec(),
        max_steps=20,
        view_radius=2,
        auto_move_explorer=auto_move_explorer,
        history_window=15,
    )


def show_messages(messages):
    for idx, msg in enumerate(messages):
        print(f'===== MESSAGE {idx} / role={msg["role"]} =====')
        print(msg['content'])
        print()


def show_state(env, include_system_prompt=True):
    state = env.get_state(include_system_prompt=include_system_prompt)
    print('phase:', state['phase'])
    print('active_player:', state['active_player'])
    print('t/max_steps:', state['t'], '/', state['max_steps'])
    print('position:', state['pos'], '| goal:', getattr(env, 'goal', None))
    print('outcome:', state['outcome'])
    print('\nfull grid:')
    print(env.render(full=True))
    print('\nlocal view:')
    print(env.render(full=False, radius=env.view_radius))
    print('\nrecent history:')
    print('\n'.join(env.history[-10:]) if env.history else '(empty)')
    print()
    show_messages(state['messages'])
    return state


In [ ]:
env = make_env(seed=0, auto_move_explorer=False)
_ = show_state(env, include_system_prompt=True)


In [ ]:
guide_action = env.get_truthful_action()
guide_result = env.manual_step(guide_action)

print('truthful guide action:')
pprint(guide_action)
print('\nguide result:')
pprint(guide_result)
print('\nnext prompt (challenge):')
_ = show_state(env, include_system_prompt=True)


In [ ]:
challenge_result = env.manual_step({'Action': 'Pass'})
print('challenge action:')
pprint({'Action': 'Pass'})
print('\nchallenge result:')
pprint(challenge_result)
print('\nstate after challenge resolution:')
_ = show_state(env, include_system_prompt=True)
